# Association rules with Python - solutions

This laboratory closely follows **13-Association rules with Weka - Solutions**, translating the Weka workflow to Python while using only the local ARFF files in `datasets/`.

We will:

- discretize the Census attributes and mine frequent itemsets;
- reproduce Weka's decreasing-support Apriori search;
- read rule support, antecedent support, confidence, lift, leverage, and conviction;
- constrain rules to a target attribute (Weka's `car` behavior);
- perform Market Basket Analysis at several support and confidence thresholds;
- use lift to interpret the link among hamburger, hamburger buns, and white bread.

> **Weka to Python:** scikit-learn does not provide Apriori or association-rule generation. We therefore use `mlxtend.frequent_patterns.apriori` and `association_rules`, whose definitions of support, confidence, lift, leverage, and conviction match Weka's. The remaining analysis follows the familiar pandas/scikit-learn workflow.

## Setup and ARFF loader

Run the notebook from either the repository root or the `slides/` directory. The path helper deliberately uses the **local** `slides/datasets` folder; no dataset is downloaded.

If needed, uncomment the installation line in the next cell.

In [ ]:
#| echo: false
#| output: false

# %pip install -q numpy pandas scipy scikit-learn mlxtend matplotlib

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import arff
from sklearn.metrics import normalized_mutual_info_score
from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False})


def find_datasets_dir():
    candidates = [Path("datasets"), Path("slides/datasets"), Path.cwd() / "slides" / "datasets"]
    for candidate in candidates:
        if (candidate / "CensusTraining.arff").exists() and (candidate / "MarketBasket.arff").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find slides/datasets. Start the notebook in the repository root or slides/.")


DATASETS = find_datasets_dir()
DATASETS


## Reusable Apriori helpers

The following utilities load ARFF data, format itemsets and rules, and reproduce Weka's decreasing-support search.

In [2]:
#| echo: false
#| output: false

def load_arff_frame(filename):
    '''Load a local ARFF file and decode byte-valued nominal columns.'''
    records, metadata = arff.loadarff(DATASETS / filename)
    frame = pd.DataFrame(records)
    for column in frame.columns:
        if frame[column].dtype == object:
            frame[column] = frame[column].map(
                lambda value: value.decode("utf-8") if isinstance(value, bytes) else value
            )
    return frame, metadata


def item_label(itemset):
    '''Stable, readable rendering of a frozenset of items.'''
    return ", ".join(sorted(map(str, itemset)))


def itemset_table(frequent_itemsets, top=None):
    table = frequent_itemsets.copy()
    table["length"] = table["itemsets"].map(len)
    table["itemset"] = table["itemsets"].map(item_label)
    table = table.sort_values(["length", "support"], ascending=[True, False])
    columns = ["length", "support", "itemset"]
    return table[columns].head(top) if top else table[columns]


RULE_COLUMNS = [
    "antecedents", "consequents", "antecedent support", "consequent support",
    "support", "confidence", "lift", "leverage", "conviction"
]


def readable_rules(rules, top=None):
    '''Format mlxtend rules like Weka's antecedent ==> consequent listing.'''
    if rules.empty:
        return pd.DataFrame(columns=["antecedent", "consequent"] + RULE_COLUMNS[2:])
    table = rules[RULE_COLUMNS].copy()
    table["antecedent"] = table.pop("antecedents").map(item_label)
    table["consequent"] = table.pop("consequents").map(item_label)
    table.insert(2, "rule length", rules["antecedents"].map(len) + rules["consequents"].map(len))
    ordered = ["antecedent", "consequent", "rule length"] + RULE_COLUMNS[2:]
    table = table[ordered]
    return table.head(top).round(4) if top else table.round(4)


def mine_rules(binary_frame, min_support, metric="confidence", min_metric=0.9, max_len=None):
    '''Mine itemsets once, then generate and rank rules using Weka-compatible metrics.'''
    itemsets = apriori(binary_frame, min_support=min_support, use_colnames=True, max_len=max_len)
    if itemsets.empty:
        return itemsets, pd.DataFrame(columns=RULE_COLUMNS)
    rules = association_rules(
        itemsets,
        num_itemsets=len(binary_frame),
        metric=metric,
        min_threshold=min_metric,
    )
    if rules.empty:
        return itemsets, rules
    rules = rules.sort_values([metric, "support", "confidence"], ascending=False).reset_index(drop=True)
    return itemsets, rules


def weka_like_search(
    binary_frame,
    lower_bound=0.10,
    upper_bound=1.00,
    delta=0.05,
    metric="confidence",
    min_metric=0.90,
    num_rules=10,
    consequent_prefix=None,
):
    '''Reproduce Weka Apriori's rounds of progressively decreasing support.'''
    support = upper_bound
    history = []
    while support >= lower_bound - 1e-12:
        itemsets, rules = mine_rules(binary_frame, support, metric, min_metric)
        if consequent_prefix is not None and not rules.empty:
            rules = rules[
                (rules["consequents"].map(len) == 1)
                & rules["consequents"].map(
                    lambda values: next(iter(values)).startswith(consequent_prefix)
                )
            ].copy()
            rules = rules.sort_values([metric, "support"], ascending=False).reset_index(drop=True)
        history.append({"min support": support, "itemsets": len(itemsets), "rules": len(rules)})
        if len(rules) >= num_rules or support <= lower_bound + 1e-12:
            break
        support = max(lower_bound, round(support - delta, 10))
    return itemsets, rules.head(num_rules), pd.DataFrame(history)

# Census data

## Dataset overview

The warm-up uses `CensusTraining.arff`, a 1,000-row sample of US census records. Its 15 attributes describe age, employment, education, household role, demographics, capital gains/losses, working hours, country, and whether total income exceeds $50K.

In [ ]:
census, census_meta = load_arff_frame("CensusTraining.arff")

print(f"Instances: {len(census):,}")
print(f"Attributes: {census.shape[1]}")


## Census sample


In [ ]:
census.head()


## Census dtypes


In [ ]:
census.dtypes.to_frame("dtype")


## Census preprocessing

Weka's association-rule search requires discrete attributes. As in the slides, we:

1. discretize every numeric attribute into **10 equal-frequency bins**;
2. remove `fnlwgt` (the final sampling weight);
3. retain one item per observed `attribute=value` pair.

Missing values are not turned into items. The helper preserves ties and places each equal-frequency cut point halfway between distinct observed values, matching Weka's behavior for attributes with many repeated zeros (notably capital gain/loss).

In [ ]:
#| echo: false
#| output: false

def discretize_census(frame, bins=10):
    result = frame.drop(columns="fnlwgt").copy()
    for column in result.select_dtypes(include="number"):
        non_missing = result[column].dropna()
        ordered = np.sort(non_missing.to_numpy())
        cut_points = []
        # Weka does not split equal values across bins. At each target rank, move the
        # boundary between that value and the next distinct observed value.
        for k in range(1, bins):
            position = min(int(np.ceil(k * len(ordered) / bins)) - 1, len(ordered) - 1)
            left = ordered[position]
            greater = ordered[ordered > left]
            if len(greater):
                cut_points.append((left + greater[0]) / 2)
        cut_points = sorted(set(cut_points))
        edges = [-np.inf, *cut_points, np.inf]
        result[column] = pd.cut(non_missing, bins=edges).astype("object")
    return result


def encode_attribute_values(frame):
    encoded = {}
    for column in frame.columns:
        values = frame[column]
        for value in sorted(values.dropna().unique(), key=str):
            encoded[f"{column}={value}"] = values.eq(value)
    return pd.DataFrame(encoded, index=frame.index, dtype=bool)


census_discrete = discretize_census(census)
census_items = encode_attribute_values(census_discrete)

print(f"After removing fnlwgt: {census_discrete.shape[1]} attributes")
print(f"One-hot transaction matrix: {census_items.shape[0]} x {census_items.shape[1]}")
census_discrete.head()


## Manual correlation check

In association analysis there is no privileged class unless we deliberately choose one. A pairwise check reveals the strongest expected relationship: `education` and `education-num` encode nearly the same concept. Normalized mutual information (NMI) is 0 for independence and 1 for a deterministic correspondence.

In [ ]:
education_pairs = census[["education", "education-num"]].dropna()
education_nmi = normalized_mutual_info_score(
    education_pairs["education"].astype(str), education_pairs["education-num"].astype(str)
)

print(f"NMI(education, education-num) = {education_nmi:.3f}")
pd.crosstab(education_pairs["education"], education_pairs["education-num"])


# Apriori

## Parameters and rule metrics

| Weka Apriori option | Python equivalent in this notebook |
|---|---|
| `car` | filter rules so the chosen class attribute is the consequent |
| `lowerBoundMinSupport` | lowest `min_support` tried |
| `upperBoundMinSupport` | first `min_support` tried |
| `delta` | decrement between support rounds |
| `metricType` | `metric` in `association_rules` |
| `minMetric` | `min_threshold` in `association_rules` |
| `numRules` | stop once enough rules exist, then return the best ones |
| `outputItemSets` | display the frequent-itemset table |

Allowed metrics include confidence, lift, leverage, and conviction. Rules are ordered by the selected metric. For a rule $A \Rightarrow B$:

- **antecedent support** is $s(A)$;
- **consequent support** is $s(B)$;
- **rule support** is $s(A \cup B)$;
- **confidence** is $s(A \cup B) / s(A)$;
- **lift** is $s(A \cup B) / (s(A)s(B))$.

Because support is anti-monotone, $s(A \cup B) \leq s(A)$ and $s(A \cup B) \leq s(B)$.

## Default search with decreasing support

Start at the upper support bound, decrease it by `delta`, and stop once ten confidence-qualified rules are available.

In [ ]:
census_itemsets_10, census_rules_10, census_history_10 = weka_like_search(
    census_items,
    lower_bound=0.10,
    upper_bound=1.00,
    delta=0.05,
    metric="confidence",
    min_metric=0.90,
    num_rules=10,
)



## Census decreasing-support rounds


In [ ]:
census_history_10


## Census frequent itemsets at the stopping round


In [ ]:
itemset_table(census_itemsets_10)


## Census best rules


In [ ]:
readable_rules(census_rules_10)


## Reading antecedent and rule support

The rule support must never exceed either side's support; the check below verifies the anti-monotone relationship numerically.

In [ ]:
if not census_rules_10.empty:
    check = census_rules_10.assign(
        anti_monotone_ok=lambda x: (x["support"] <= x["antecedent support"] + 1e-12)
        & (x["support"] <= x["consequent support"] + 1e-12)
    )[["antecedent support", "consequent support", "support", "anti_monotone_ok"]]
    assert check["anti_monotone_ok"].all()

check.round(4) if not census_rules_10.empty else pd.DataFrame()


## Iterative search with decreasing support

Weka starts at the upper support bound and lowers it by `delta`. Each round returns rules above `minMetric`; the cycle stops when it reaches the lower bound or has found `numRules` rules. Increasing `numRules` therefore permits more rounds and can reveal lower-support but higher-confidence rules.

In [ ]:
census_itemsets_100, census_rules_100, census_history_100 = weka_like_search(
    census_items,
    lower_bound=0.10,
    upper_bound=1.00,
    delta=0.05,
    metric="confidence",
    min_metric=0.90,
    num_rules=100,
)

print(f"Stopping support: {census_history_100.iloc[-1]['min support']:.2f}")
print(f"Rules returned: {len(census_rules_100)}")


## Census broad rule search history


In [ ]:
census_history_100


## Census broad rule search results


In [ ]:
readable_rules(census_rules_100, top=15)


## Interpreting the broad rule search

The first returned rules maximize confidence, even when their support is lower. Limiting the support interval is useful when we want patterns at a particular prevalence and want to avoid rules driven only by extremely common single attributes.

## Calibrating support for rules of length 3

The reference solution suggests a lower support near **0.70**. At that level, only very frequent items survive, so the remaining 3-item sets generate several different rules while longer, rarer combinations disappear.

In [ ]:
census_itemsets_70, census_rules_70 = mine_rules(
    census_items, min_support=0.70, metric="confidence", min_metric=0.90
)

length_summary_70 = (
    census_itemsets_70.assign(length=census_itemsets_70["itemsets"].map(len))
    .groupby("length").size().rename("frequent itemsets").to_frame()
)
print(f"Frequent itemsets: {len(census_itemsets_70)}")
print(f"Rules generated: {len(census_rules_70)}")


## Census high-support itemset lengths


In [ ]:
length_summary_70


## Census high-support rules


In [ ]:
readable_rules(census_rules_70)


## Why rules can outnumber itemsets

Each frequent itemset can produce multiple antecedent/consequent partitions, which is why the number of rules may exceed the number of itemsets. At a high support bound, the search can also stop because no further itemsets satisfy both the support and confidence thresholds.

## Rules for a target attribute (`car=True`)

Weka's `car=True` constrains the chosen class attribute to the consequent. Here the original class (attribute 15 in Python's one-based counting; Weka may show a zero-based index) is represented by items beginning with `class=`.

In [ ]:
car_itemsets, car_rules, car_history = weka_like_search(
    census_items,
    lower_bound=0.10,
    upper_bound=1.00,
    delta=0.05,
    metric="confidence",
    min_metric=0.90,
    num_rules=20,
    consequent_prefix="class=",
)



## Census class-consequent search history


In [ ]:
car_history


## Census class-consequent rules


In [ ]:
readable_rules(car_rules)


## Census class-consequent rule counts


In [ ]:
car_rules["consequents"].map(item_label).value_counts().to_frame("rules") if not car_rules.empty else pd.DataFrame()


## Interpreting class-association rules

All returned rules have income class as the consequent. The dominant consequent is `class=<=50K`, because that is by far the more frequent class in this sample. Weka-style CAR search constrains the *attribute*, not a specific value; filtering to one value is an additional step:

In [ ]:
target_over_50k = car_rules[
    car_rules["consequents"].map(lambda values: "class=>50K" in values)
]
readable_rules(target_over_50k)


# Market Basket Analysis

## Dataset overview

Market Basket Analysis identifies buying habits that can inform promotions, shelf placement, and advertising. `MarketBasket.arff` contains 651 transactions and 56 binary product attributes.

The local file uses Weka's dense asymmetric encoding: `t` means purchased, while `f` or `?` means absent. Treating absence as a regular item would produce unhelpful rules about products *not* being bought, so only `t` becomes `True` in the transaction matrix.

In [ ]:
market_raw, market_meta = load_arff_frame("MarketBasket.arff")
market_raw.columns = market_raw.columns.str.strip()
market = market_raw.eq("t").astype(bool)

print(f"Instances: {len(market):,}")
print(f"Products: {market.shape[1]}")
print(f"Average products per transaction: {market.sum(axis=1).mean():.2f}")
market.head()


## Dense and sparse ARFF formats

With one attribute per product, each row is a transaction. Weka supports:

```text
% Dense: missing means not purchased
@attribute product1 {t}
@attribute product2 {t}
@data
?,t
t,?
```

```text
% Sparse: include only purchased product index/value pairs
@attribute product1 {f,t}
@attribute product2 {f,t}
@data
{1 t}
{0 t, 1 t}
```

Both map to the same Boolean transaction matrix used below.

## Threshold experiments

We reproduce the threshold experiments from the reference:

1. default-like settings: support 0.10, 10 rules, confidence 0.90;
2. support 0.01, confidence 0.90;
3. support 0.05, confidence 0.90;
4. support 0.05, confidence 0.70.

In [ ]:
#| echo: false

market_experiments = []
market_results = {}

for min_support, min_confidence in [(0.10, 0.90), (0.01, 0.90), (0.05, 0.90), (0.05, 0.70)]:
    itemsets, rules = mine_rules(
        market, min_support=min_support, metric="confidence", min_metric=min_confidence
    )
    key = (min_support, min_confidence)
    market_results[key] = (itemsets, rules)
    market_experiments.append({
        "min support": min_support,
        "min confidence": min_confidence,
        "frequent itemsets": len(itemsets),
        "maximum itemset length": int(itemsets["itemsets"].map(len).max()) if len(itemsets) else 0,
        "rules": len(rules),
    })

market_experiments = pd.DataFrame(market_experiments)
market_experiments


## Rules returned by each threshold

Compare the highest-ranked rules after changing minimum support and minimum confidence.

## Market rules with default thresholds


In [ ]:
itemsets, rules = market_results[(0.10, 0.90)]
print("No association rules found." if rules.empty else f"Rules: {len(rules)}")
readable_rules(rules, top=10) if not rules.empty else pd.DataFrame()


## Market rules with lower support


In [ ]:
itemsets, rules = market_results[(0.01, 0.90)]
print("No association rules found." if rules.empty else f"Rules: {len(rules)}")
readable_rules(rules, top=10) if not rules.empty else pd.DataFrame()


## Market rules with moderate support


In [ ]:
itemsets, rules = market_results[(0.05, 0.90)]
print("No association rules found." if rules.empty else f"Rules: {len(rules)}")
readable_rules(rules, top=10) if not rules.empty else pd.DataFrame()


## Market rules with lower confidence


In [ ]:
itemsets, rules = market_results[(0.05, 0.70)]
print("No association rules found." if rules.empty else f"Rules: {len(rules)}")
readable_rules(rules, top=10) if not rules.empty else pd.DataFrame()


## Interpreting the threshold experiments

At support 0.10 the frequent-itemset collection is short and no rule reaches 0.90 confidence. Lowering support to 0.01 admits long, rare combinations; their small antecedent counts make confidence 1 possible by chance. At support 0.05, confidence 0.90 is still too strict. Reducing confidence to 0.70 yields shorter, better-supported rules, but confidence alone still favors very common consequent products.

## Why confidence can be misleading

Frequently purchased staples can appear in high-confidence rules even without a strong association. The reference highlights 2% milk, eggs, cola, and white bread.

In [ ]:
staples = ["2pct. Milk", "Eggs", "Cola", "White Bread"]
staple_support = market[staples].mean().sort_values(ascending=False).to_frame("support")


## Market staple support table


In [ ]:
staple_support.round(3)


## Market staple support chart


In [ ]:
ax = staple_support.sort_values("support").plot.barh(legend=False, color="#4C9BC6")
ax.set(xlabel="support", ylabel="", title="Support of frequently purchased products")
plt.show()


## From frequent staples to target rules

These products are bought often regardless of other items. Rare combinations can therefore imply a staple with high confidence while having very low support. Raising support removes many accidental combinations, but we then need to accept lower confidence and shorter patterns.

## Rules with a specific consequent: Eggs

To imitate a target-item search, retain only rules whose consequent is exactly `Eggs`. The high-confidence rules are based on small joint counts, confirming that eggs are frequent but not tightly linked to one particular basket.

In [ ]:
market_itemsets_01, market_rules_01 = market_results[(0.01, 0.90)]
egg_rules = market_rules_01[
    market_rules_01["consequents"].map(lambda values: values == frozenset({"Eggs"}))
].copy()

readable_rules(egg_rules, top=15)


# Lift

## Use lift to reveal correlation

Confidence $P(B\mid A)$ does not correct for how common $B$ already is. Lift does:

$$
\operatorname{lift}(A \Rightarrow B) = \frac{P(A \cap B)}{P(A)P(B)}
$$

- lift $>1$: positive association;
- lift $=1$: independence;
- lift $<1$: negative association.

Using support 0.10 keeps the comparison focused on well-supported product pairs.

In [ ]:
market_itemsets_lift, market_rules_lift = mine_rules(
    market, min_support=0.10, metric="lift", min_metric=1.0
)

readable_rules(market_rules_lift, top=15)


## Hamburger buns, fat-free hamburger, and white bread

The strongest lifted pair is `98pct. Fat Free Hamburger` with `Hamburger Buns`; fat-free hamburger is also positively associated with `White Bread`. We now inspect the exact counts, including the three-way overlap that is hidden by pairwise rules.

In [ ]:
hamburger = "98pct. Fat Free Hamburger"
buns = "Hamburger Buns"
bread = "White Bread"

counts = pd.Series({
    "Hamburger": int(market[hamburger].sum()),
    "Buns": int(market[buns].sum()),
    "White bread": int(market[bread].sum()),
    "Hamburger and buns": int((market[hamburger] & market[buns]).sum()),
    "Hamburger and white bread": int((market[hamburger] & market[bread]).sum()),
    "Hamburger, buns, and white bread": int((market[hamburger] & market[buns] & market[bread]).sum()),
}, name="transactions")


## Hamburger basket counts


In [ ]:
counts.to_frame()


## Hamburger bun and bread choices


In [ ]:
hamburger_baskets = market.loc[market[hamburger], [buns, bread]]
choice_table = (
    hamburger_baskets.value_counts()
    .rename("transactions")
    .rename_axis(["buys buns", "buys white bread"])
    .to_frame()
)
choice_table


## Pairwise lift rules for the three products

Filter the lifted rules to the hamburger, buns, and white-bread subset before interpreting the relationship.

In [ ]:
focus_rules = market_rules_lift[
    market_rules_lift["antecedents"].map(lambda s: s.issubset({hamburger, buns, bread}))
    & market_rules_lift["consequents"].map(lambda s: s.issubset({hamburger, buns, bread}))
]
readable_rules(focus_rules)


## Solution and caveat

**Solution.** Buyers of the fat-free hamburger often pair it with either white bread or hamburger buns, and both pairwise associations have lift well above 1. The three-way combination has support only $36/651 \approx 0.055$, so it is absent from the itemsets at the chosen 0.10 support threshold. This supports the slide's interpretation of the two bread products as alternatives at that threshold, but the direct count also shows that they are **not literally mutually exclusive**. Pairwise rules alone cannot establish exclusivity.

# Takeaways

## Association-rule checklist

- Apriori exploits the anti-monotone property of support to prune itemsets.
- Weka's decreasing-support loop can be reproduced by repeated `mlxtend` Apriori calls.
- High confidence is not automatically interesting when the consequent is already common.
- Support controls prevalence; confidence controls conditional reliability; lift measures departure from independence.
- Target-consequent filtering reproduces class association rules, but specifying one target value requires an extra filter.
- Inspect the original counts before turning pairwise rules into a business conclusion.